In [12]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient

# --- 1. INITIALIZATION & HIGH-VISIBILITY STYLING ---
OUT_DIR = "./figures/executive_clear_set"
os.makedirs(OUT_DIR, exist_ok=True)

# Standardizing for high-visibility: White background, large fonts
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 16, 'axes.labelsize': 14})
palette = sns.color_palette("bright")

print("Connecting to MongoDB and preparing data...")
client = MongoClient("mongodb://localhost:27017")
col = client["amazon_reviews"]["reviews"]

# Efficient data loading
cursor = col.find({}, {"ReviewDate": 1, "Score": 1, "SentimentLabel": 1, 
                       "HelpfulnessRatio": 1, "UserId": 1, "Text": 1, "_id": 0})
df = pd.DataFrame(list(cursor))
df["ReviewDate"] = pd.to_datetime(df["ReviewDate"], errors="coerce")
df["Year"] = df["ReviewDate"].dt.year
df = df[(df["Year"] >= 2000) & (df["Year"] <= 2012)]

# --- FIGURE A: MACRO TIMELINE (Growth Context) ---
print("Plotting Figure A: Macro Timeline...")
yearly_counts = df.groupby("Year").size()
plt.figure(figsize=(12, 6))
plt.plot(yearly_counts.index, yearly_counts.values, marker="o", color=palette[0], linewidth=3)
plt.fill_between(yearly_counts.index, yearly_counts.values, alpha=0.1, color=palette[0])

# Major Events
events = {2005: "Amazon Prime", 2007: "Amazon Fresh", 2011: "Mobile Boom"}
for yr, txt in events.items():
    plt.axvline(x=yr, color="#333333", linestyle="--", alpha=0.7)
    plt.text(yr, yearly_counts.max()*0.85, txt, ha="center", weight="bold", bbox=dict(facecolor='white', alpha=0.8))

plt.title("Total Review Volume: The Impact of Industry Milestones")
plt.xlabel("Year")
plt.ylabel("Review Count")
plt.savefig(f"{OUT_DIR}/figA_macro_timeline.png", dpi=300)
plt.close()

# --- FIGURE B: LOGISTICS IMPACT (The Story of Service Reliability) ---
print("Plotting Figure B: Logistics Analysis...")
themes = {"Prime & Shipping": "shipping|delivery|fast", "Subscribe & Save": "subscribe|cancel|stale"}
plt.figure(figsize=(12, 6))

for i, (theme, regex) in enumerate(themes.items()):
    theme_df = df[df["Text"].str.contains(regex, case=False, na=False)]
    yearly_totals = theme_df.groupby("Year").size()
    neg_totals = theme_df[theme_df["SentimentLabel"] == "Negative"].groupby("Year").size()
    neg_rate = (neg_totals / yearly_totals).fillna(0) * 100
    plt.plot(neg_rate.index, neg_rate.values, marker="s", label=theme, linewidth=3, color=palette[i+1])

plt.title("Negative Sentiment Rate: The Success of Logistics Programs")
plt.ylabel("Negative Reviews (%)")
plt.ylim(0, 50)
plt.legend(frameon=True, shadow=True)
plt.savefig(f"{OUT_DIR}/figB_logistics_impact.png", dpi=300)
plt.close()

# --- FIGURE C: THE BLUE OCEAN (Market Opportunities) ---
print("Plotting Figure C: Health Trends...")
health_regex = "organic|natural|gluten|vegan|dairy free"
health_df = df[(df["Score"] == 5) & (df["Text"].str.contains(health_regex, case=False, na=False))]
health_trend = health_df.groupby("Year").size()

plt.figure(figsize=(12, 6))
plt.plot(health_trend.index, health_trend.values, color=palette[2], marker="^", linewidth=3)
plt.title("Market Opportunity: Exponential Growth in 5-Star Health Reviews")
plt.ylabel("Count of 5-Star Reviews")
plt.savefig(f"{OUT_DIR}/figC_health_boom.png", dpi=300)
plt.close()

# --- FIGURE D: SENTIMENT PARADOX (Trust Visualization) ---
print("Plotting Figure D: Helpfulness Density...")
df_trust = df[df["HelpfulnessRatio"].notna()]
plt.figure(figsize=(10, 6))
hb = plt.hexbin(df_trust["Score"], df_trust["HelpfulnessRatio"], gridsize=15, cmap='YlOrRd', mincnt=1)
plt.colorbar(hb, label='Number of Reviews')
plt.title("Trust Distribution: Helpfulness Density across Star Ratings")
plt.xlabel("Star Rating")
plt.ylabel("Helpfulness Ratio (0 to 1)")
plt.savefig(f"{OUT_DIR}/figD_trust_density.png", dpi=300)
plt.close()

# --- FIGURE E: LOYALTY AUDIT (User Archetypes) ---
print("Plotting Figure E: User Distribution...")
user_counts = df["UserId"].value_counts()
power_users = df[df["UserId"].isin(user_counts[user_counts >= 10].index)]
casual_users = df[df["UserId"].isin(user_counts[user_counts == 1].index)]

plt.figure(figsize=(10, 6))
sns.kdeplot(power_users["Score"], label="Power Users (10+ Reviews)", color=palette[0], fill=True, alpha=0.3, linewidth=3)
sns.kdeplot(casual_users["Score"], label="Casual Users (1 Review)", color=palette[3], fill=True, alpha=0.3, linewidth=3)
plt.title("Rating Bias: Comparison of Power Users vs. Casual Customers")
plt.xlabel("Star Rating")
plt.ylabel("Density of Ratings")
plt.legend()
plt.savefig(f"{OUT_DIR}/figE_loyalty_distribution.png", dpi=300)
plt.close()

print(f"Master script complete. 5 High-Visibility PNGs saved to: {OUT_DIR}")
client.close()

Connecting to MongoDB and preparing data...
Plotting Figure A: Macro Timeline...
Plotting Figure B: Logistics Analysis...
Plotting Figure C: Health Trends...
Plotting Figure D: Helpfulness Density...
Plotting Figure E: User Distribution...
Master script complete. 5 High-Visibility PNGs saved to: ./figures/executive_clear_set


In [13]:
import os
import time
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient

# --- 1. CONFIGURATION & STYLING ---
OUT_DIR = "./final_project_submission"
os.makedirs(OUT_DIR, exist_ok=True)

# Projector-ready styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'figure.facecolor': 'white'})
palette = sns.color_palette("bright")

WORD_LIST = [
    "delicious", "organic", "shipping", "stale", "cheap", 
    "perfect", "packaging", "expire", "water", "coffee", 
    "bulk", "natural", "fast", "arrived", "cancel", 
    "broken", "box", "delivery", "fresh", "quality"
]

print("Connecting to MongoDB...")
client = MongoClient("mongodb://localhost:27017")
col = client["amazon_reviews"]["reviews"]

# --- 2. PERFORMANCE BENCHMARKING FUNCTIONS ---

def benchmark_pandas(words):
    print("  [Method A] Running Pandas Client-Side Benchmark...")
    start_time = time.perf_counter()
    cursor = col.find({}, {"Text": 1, "_id": 0})
    df_temp = pd.DataFrame(list(cursor))
    
    for word in words:
        _ = df_temp['Text'].str.contains(word, case=False, na=False).sum()
    
    end_time = time.perf_counter()
    mem_use = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
    return (end_time - start_time) / len(words), mem_use

def benchmark_regex(words):
    print("  [Method B] Running MongoDB Regex Benchmark...")
    start_time = time.perf_counter()
    for word in words:
        _ = col.count_documents({"Text": {"$regex": word, "$options": "i"}})
    end_time = time.perf_counter()
    mem_use = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
    return (end_time - start_time) / len(words), mem_use

def benchmark_aggregation(words):
    print("  [Method C] Running MongoDB Aggregation Benchmark...")
    start_time = time.perf_counter()
    for word in words:
        pipeline = [{"$match": {"Text": {"$regex": word, "$options": "i"}}}, {"$group": {"_id": None, "count": {"$sum": 1}}}]
        _ = list(col.aggregate(pipeline))
    end_time = time.perf_counter()
    mem_use = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
    return (end_time - start_time) / len(words), mem_use

# --- 3. EXECUTE BENCHMARK & PLOT EFFICIENCY ---
print("\nStarting Performance Comparison (Counting 20 random words)...")
time_a, mem_a = benchmark_pandas(WORD_LIST)
time_b, mem_b = benchmark_regex(WORD_LIST)
time_c, mem_c = benchmark_aggregation(WORD_LIST)

bench_df = pd.DataFrame({
    "Method": ["Pandas (Client)", "Regex (Server)", "Aggregation (Server)"],
    "Avg Time (s)": [time_a, time_b, time_c],
    "Local RAM (MB)": [mem_a, mem_b, mem_c]
})

fig, ax1 = plt.subplots(figsize=(10, 6))
# Fixed FutureWarnings by assigning hue and setting legend=False
sns.barplot(x="Method", y="Avg Time (s)", data=bench_df, ax=ax1, palette="Blues_d", hue="Method", legend=False)
ax2 = ax1.twinx()
sns.lineplot(x="Method", y="Local RAM (MB)", data=bench_df, ax=ax2, color="red", marker="o", linewidth=3)
plt.title("System Efficiency: Local RAM vs. Execution Time")
plt.savefig(f"{OUT_DIR}/efficiency_audit.png", dpi=300)
plt.show() # Shows plot inline

# --- 4. DATA INSIGHT VISUALIZATIONS ---
print("\nGenerating Actionable Insight Figures...")
cursor = col.find({}, {"ReviewDate": 1, "Score": 1, "SentimentLabel": 1, "HelpfulnessRatio": 1, "Text": 1, "_id": 0})
df = pd.DataFrame(list(cursor))
df["ReviewDate"] = pd.to_datetime(df["ReviewDate"], errors="coerce")
df["Year"] = df["ReviewDate"].dt.year
df = df[(df["Year"] >= 2000) & (df["Year"] <= 2012)]

# FIGURE 1: LOGISTICS ROI
plt.figure(figsize=(10, 6))
shipping_df = df[df["Text"].str.contains("shipping|delivery|arrived", case=False, na=False)]
neg_rate = (shipping_df[shipping_df["SentimentLabel"] == "Negative"].groupby("Year").size() / shipping_df.groupby("Year").size()).fillna(0) * 100
plt.plot(neg_rate.index, neg_rate.values, marker='o', linewidth=3, color=palette[0], label="Shipping Dissatisfaction")
plt.axvline(x=2005, color='gray', linestyle='--')
plt.text(2005.2, plt.gca().get_ylim()[1]*0.8, "Amazon Prime Launch", fontweight='bold', bbox=dict(facecolor='white', alpha=0.7))
plt.title("Logistics Impact: Prime Success in Reducing Shipping Complaints")
plt.ylabel("Negative Review Rate (%)")
plt.savefig(f"{OUT_DIR}/fig1_logistics_success.png", dpi=300)
plt.show()

# FIGURE 2: MARKET GROWTH (HEALTH)
plt.figure(figsize=(10, 6))
health_df = df[df["Text"].str.contains("organic|natural|gluten free|vegan", case=False, na=False)]
health_growth = health_df[health_df["Score"] == 5].groupby("Year").size()
plt.bar(health_growth.index, health_growth.values, color=palette[2], alpha=0.6)
plt.plot(health_growth.index, health_growth.values, marker='s', color=palette[2], linewidth=2)
plt.title("Market Direction: Rapid Expansion of 5-Star 'Health & Organic' Reviews")
plt.savefig(f"{OUT_DIR}/fig2_health_opportunity.png", dpi=300)
plt.show()

# FIGURE 3: TRUST FACTOR
plt.figure(figsize=(10, 6))
# Fixed FutureWarnings by assigning hue and setting legend=False
sns.boxplot(x="Score", y="HelpfulnessRatio", data=df[df["HelpfulnessRatio"] > 0], palette="flare", hue="Score", legend=False)
plt.title("Consumer Trust: Community Reliance on Polarized Ratings (1 vs 5 Stars)")
plt.savefig(f"{OUT_DIR}/fig3_trust_distribution.png", dpi=300)
plt.show()

client.close()
print(f"\nAll tasks complete. Audit saved to: {OUT_DIR}")

Connecting to MongoDB for high-efficiency data extraction...
Generating Plot 1: Logistics ROI...
Generating Plot 2: Market Opportunity...
Generating Plot 3: Helpfulness Distribution...


/tmp/ipykernel_20906/971209928.py:76: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxenplot(x="Score", y="HelpfulnessRatio", data=df[df["HelpfulnessRatio"] > 0], palette="viridis")


Success! All complementary plots saved to ./final_submission_figures
